# Telugu Text Classifier — Phase 2 training (Colab GPU)

Fine-tunes IndicBERTv2 / MuRIL / XLM-R on the frozen Phase 1 splits.

**Before running:** `Runtime > Change runtime type > T4 GPU`, and set `REPO_URL` below (the repo must be pushed to GitHub).
Total: 9 seed runs + 1 LR comparison ≈ 2–2.5 h on a T4. Checkpoints land on Google Drive so they survive disconnects.

In [1]:
REPO_URL = "https://github.com/mahendravelagapudi099-wq/telugu-text-classifier.git"  # <-- EDIT
DRIVE_DIR = "/content/drive/MyDrive/telugu-classifier"               # checkpoints + results land here

In [ ]:
import shutil, subprocess, torch

# Check nvidia-smi if available on PATH
if shutil.which('nvidia-smi'):
    res = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True)
    print(res.stdout)
else:
    print('nvidia-smi utility not found in system PATH.')

if torch.cuda.is_available():
    print('✅ GPU detected:', torch.cuda.get_device_name(0))
else:
    print('❌ No GPU detected! Currently running on CPU.')
    print('To enable T4 GPU in Google Colab:')
    print('1. In the top-right toolbar of this notebook editor, click the Kernel selector / Colab icon.')
    print('2. Select "Connect to Google Colab" and choose "T4 GPU".')
    print('   (Or in Colab Web: Runtime -> Change runtime type -> T4 GPU)')
    assert torch.cuda.is_available(), 'Please connect to a T4 GPU runtime before running transformer training.'


In [ ]:
# Unpack local repository files and baseline test predictions directly into Colab
import base64, gzip, json, pathlib
ROOT = pathlib.Path('/content/Telugu-Text-Classifier')
files = json.loads(gzip.decompress(base64.b64decode('...')))
for p, b in files.items():
    path = ROOT / p
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(base64.b64decode(b))
print('Successfully unpacked', len(files), 'files into', ROOT)


In [ ]:
import os
# Navigate to repository root
if not os.path.exists('/content/Telugu-Text-Classifier'):
    !git clone {REPO_URL} /content/Telugu-Text-Classifier
%cd /content/Telugu-Text-Classifier
# Install project dependencies
!pip install -q -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)
os.makedirs("results/runs", exist_ok=True)
print("checkpoints ->", f"{DRIVE_DIR}/outputs")

In [ ]:
# Checkpoint root lives on Drive: survives runtime disconnects
!rm -rf outputs && ln -s "{DRIVE_DIR}/outputs" outputs
# Reproduces data/processed/{train,validation,test}.csv exactly (seeded Phase 1 pipeline)
!python scripts/prepare_data.py

In [ ]:
# 3 models x 3 seeds grid (sequential)
# To see live progress in the cell, we run python scripts/run_experiments.py directly.
!python scripts/run_experiments.py


In [ ]:
!tail -5 train.log
!ls results/runs/ 2>/dev/null

In [ ]:
# After all 9 runs: select best model on VALIDATION macro-F1, then run LR comparison at 5e-5
import json, glob, pathlib, statistics, subprocess
by_model = {}
for p in glob.glob('results/runs/*.json'):
    s = json.loads(pathlib.Path(p).read_text(encoding='utf-8'))
    m_name = s['model'].split('/')[-1]
    by_model.setdefault(m_name, []).append(s['best_val_macro_f1'])
means = {m: statistics.mean(v) for m, v in by_model.items()}
best_short_name = max(means, key=means.get)
print('Mean Validation Macro-F1 across seeds:', {m: round(v, 4) for m, v in means.items()})
print('Best model (validation-selected):', best_short_name)
# Find full model id
runs = [json.loads(pathlib.Path(p).read_text(encoding='utf-8')) for p in glob.glob('results/runs/*.json')]
best_full_id = next(r['model'] for r in runs if r['model'].endswith(best_short_name))
print(f'Running LR comparison (lr=5e-5, seed=42) for {best_full_id}...')
!python scripts/run_experiments.py --models {best_full_id} --seeds 42 --lr 5e-5


In [ ]:
# Test-set evaluation, once per final model architecture:
# Best seed (by val macro-F1) at the best LR (by val macro-F1).
import json, glob, pathlib, subprocess
runs = {}
for p in glob.glob('results/runs/*.json'):
    s = json.loads(pathlib.Path(p).read_text(encoding='utf-8'))
    m_name = s['model'].split('/')[-1]
    s['short_name'] = m_name
    if 'lr' not in s and 'learning_rate' in s:
        s['lr'] = s['learning_rate']
    runs.setdefault(m_name, []).append(s)

for name, rs in sorted(runs.items()):
    best_run = max(rs, key=lambda r: r['best_val_macro_f1'])
    lr_val = best_run.get('lr', best_run.get('learning_rate', 2e-5))
    seed_val = best_run['seed']
    lr_suffix = f'_lr{lr_val:g}' if lr_val != 2e-5 else ''
    model_dir = f'outputs/{name}_seed{seed_val}{lr_suffix}'
    print(f'>>> Evaluating {name} on test set (best val run: seed={seed_val}, lr={lr_val}) from {model_dir}')
    r = subprocess.run(['python', 'scripts/eval_transformer.py',
                        '--model-dir', model_dir,
                        '--name', name], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())


In [ ]:
# Generate comparison table, paired bootstrap test, side-by-side confusion matrices, and error analysis
!python scripts/compare_models.py
!python scripts/export_errors_analysis.py
# Copy results and artifacts to Google Drive
import os
if os.path.exists('/content/drive/MyDrive'):
    !cp -r results/transformer_*_test_metrics.json results/runs/*.json results/comparison_* "{DRIVE_DIR}/"
    !cp -r results/error_analysis_* "{DRIVE_DIR}/"
    !cp -r reports/comparison.md "{DRIVE_DIR}/"
    print('Saved artifacts to Google Drive:', DRIVE_DIR)
!ls -l results/
